In [1]:
import numpy as np

# ==========================================
# 1. ENVIRONMENT & PROBLEM SETUP
# ==========================================
START = np.array([0.0, 0.0])
TARGET = np.array([10.0, 10.0])
NUM_WAYPOINTS = 5

# Enclosing wall coordinates
X_MIN, X_MAX = 0.0, 10.0
Y_MIN, Y_MAX = 0.0, 10.0

# Circular Obstacles: [x_center, y_center, radius]
OBSTACLES = [
    np.array([3.0, 3.0, 1.5]),
    np.array([5.0, 6.0, 1.8]),
    np.array([7.0, 4.0, 1.2])
]

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def line_intersects_circle(p1, p2, circle):
    """Checks if a line segment between p1 and p2 intersects a circular obstacle."""
    cx, cy, r = circle
    c = np.array([cx, cy])

    d = p2 - p1
    f = p1 - c

    a = np.dot(d, d)
    # Safe check for overlapping/identical consecutive waypoints
    if a == 0:
        return np.linalg.norm(p1 - c) <= r

    b = 2 * np.dot(f, d)
    cc = np.dot(f, f) - r**2

    discriminant = b**2 - 4*a*cc
    if discriminant < 0:
        return False

    discriminant = np.sqrt(discriminant)
    t1 = (-b - discriminant) / (2*a)
    t2 = (-b + discriminant) / (2*a)

    if (0 <= t1 <= 1) or (0 <= t2 <= 1):
        return True
    return False

def reconstruct_path(nest):
    waypoints = nest.reshape((NUM_WAYPOINTS, 2))
    return np.vstack([START, waypoints, TARGET])

def fitness_function(nest):
    path = reconstruct_path(nest)
    total_distance = 0.0
    penalty = 0.0

    for i in range(len(path) - 1):
        p1 = path[i]
        p2 = path[i+1]

        total_distance += np.linalg.norm(p2 - p1)

        for obs in OBSTACLES:
            if line_intersects_circle(p1, p2, obs):
                penalty += 1000.0  # Heavy penalty for hitting walls/obstacles

    return total_distance + penalty

# ==========================================
# 3. GREY WOLF OPTIMIZER (GWO)
# ==========================================
def grey_wolf_optimizer(pop_size=40, max_iter=300):
    dim = NUM_WAYPOINTS * 2

    # Initialize Alpha, Beta, and Delta structures
    alpha_pos = np.zeros(dim)
    alpha_score = float("inf")

    beta_pos = np.zeros(dim)
    beta_score = float("inf")

    delta_pos = np.zeros(dim)
    delta_score = float("inf")

    # Initialize random starting positions for the wolves
    positions = np.zeros((pop_size, dim))
    for i in range(pop_size):
        positions[i, 0::2] = np.random.uniform(X_MIN, X_MAX, NUM_WAYPOINTS)
        positions[i, 1::2] = np.random.uniform(Y_MIN, Y_MAX, NUM_WAYPOINTS)

    # Main Optimization Loop
    for l in range(max_iter):
        for i in range(pop_size):
            # Enforce boundary constraints
            positions[i, 0::2] = np.clip(positions[i, 0::2], X_MIN, X_MAX)
            positions[i, 1::2] = np.clip(positions[i, 1::2], Y_MIN, Y_MAX)

            # Calculate fitness
            fitness = fitness_function(positions[i])

            # Update Alpha, Beta, and Delta hierarchies
            if fitness < alpha_score:
                alpha_score = fitness
                alpha_pos = positions[i].copy()
            elif fitness < beta_score:
                beta_score = fitness
                beta_pos = positions[i].copy()
            elif fitness < delta_score:
                delta_score = fitness
                delta_pos = positions[i].copy()

        # Linearly decrease parameter 'a' from 2 to 0
        a = 2 - l * (2 / max_iter)

        # Update the position of all wolves (Omega wolves)
        for i in range(pop_size):
            for j in range(dim):
                # Calculations relative to Alpha wolf
                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * alpha_pos[j] - positions[i, j])
                X1 = alpha_pos[j] - A1 * D_alpha

                # Calculations relative to Beta wolf
                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * beta_pos[j] - positions[i, j])
                X2 = beta_pos[j] - A2 * D_beta

                # Calculations relative to Delta wolf
                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * delta_pos[j] - positions[i, j])
                X3 = delta_pos[j] - A3 * D_delta

                # Take the average to update position
                positions[i, j] = (X1 + X2 + X3) / 3

    return alpha_pos, alpha_score

# ==========================================
# 4. EXECUTION
# ==========================================
if __name__ == "__main__":
    best_solution, best_score = grey_wolf_optimizer(pop_size=40, max_iter=300)
    optimized_path = reconstruct_path(best_solution)

    print("--- ENVIRONMENT CONFIGURATION ---")
    print(f"Wall Boundaries: X limits [{X_MIN:.1f}, {X_MAX:.1f}], Y limits [{Y_MIN:.1f}, {Y_MAX:.1f}]")
    print("Obstacles Coordinates (Center X, Center Y, Radius):")
    for idx, obs in enumerate(OBSTACLES):
        print(f"  Obstacle {idx + 1}: Center=[{obs[0]:.2f}, {obs[1]:.2f}], Radius={obs[2]:.2f}")

    print("\n--- OPTIMIZATION RESULTS ---")
    print(f"Final Path Score/Fitness (GWO): {best_score:.4f}")
    print("\nOptimized Waypoint Sequence:")
    for idx, point in enumerate(optimized_path):
        if idx == 0:
            print(f"  Start:  [{point[0]:.2f}, {point[1]:.2f}]")
        elif idx == len(optimized_path) - 1:
            print(f"  Target: [{point[0]:.2f}, {point[1]:.2f}]")
        else:
            print(f"  WP {idx}:  [{point[0]:.2f}, {point[1]:.2f}]")


--- ENVIRONMENT CONFIGURATION ---
Wall Boundaries: X limits [0.0, 10.0], Y limits [0.0, 10.0]
Obstacles Coordinates (Center X, Center Y, Radius):
  Obstacle 1: Center=[3.00, 3.00], Radius=1.50
  Obstacle 2: Center=[5.00, 6.00], Radius=1.80
  Obstacle 3: Center=[7.00, 4.00], Radius=1.20

--- OPTIMIZATION RESULTS ---
Final Path Score/Fitness (GWO): 15.1460

Optimized Waypoint Sequence:
  Start:  [0.00, 0.00]
  WP 1:  [0.00, 0.03]
  WP 2:  [0.11, 0.76]
  WP 3:  [0.12, 0.90]
  WP 4:  [0.34, 1.42]
  WP 5:  [3.74, 7.43]
  Target: [10.00, 10.00]
